# DAG Parameterization – Year Sensitivity Analysis
Evaluates whether model performance is stable across the three cohort years in the test set (2020, 2021, 2022).
The test set covers patients admitted on or after 2020; temporal stability suggests the model generalises without year-specific drift.

In [ ]:
# Data manipulation and plotting
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import re
import glob

# Metrics
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, balanced_accuracy_score
from sklearn.calibration import calibration_curve
from pycalib.metrics import ECE

import xgboost

## Data preparation

In [ ]:
# Load training data
X_train = pd.read_csv('../../Data Pre-processing/X_train_c12_w48_imp.csv', index_col=0)
X_test  = pd.read_csv('../../Data Pre-processing/X_test_c12_w48_imp.csv',  index_col=0)

print(X_train.shape, X_test.shape)

# Remove features with near-zero variance
X_train = X_train.loc[:, X_train.var() >= 0.01]
X_test  = X_test.filter(items=X_train.columns)
print(X_train.shape, X_test.shape)

# MICE-imputed training set (used for DAG feature selection)
X_train_imp = pd.read_csv('../../Data Pre-processing/X_train_c12_w48_mice.csv', index_col=0)

y_train = pd.read_csv('../../Data Pre-processing/y_train_c12_w48_imp.csv', index_col=[0,1]).groupby('uid').max()
y_test  = pd.read_csv('../../Data Pre-processing/y_test_c12_w48_imp.csv',  index_col=[0,1]).groupby('uid').max()

# Correlation-based feature selection (threshold used in original experiments)
correlation_threshold = 0.3
X_train_corr = X_train_imp.loc[:, X_train_imp.corrwith(y_train['Outcome']).abs() >= correlation_threshold]
print('Correlation-filtered features:', X_train_corr.shape)

## Year-split test sets
Join admission year from `patients.parquet` to the test UIDs and partition into 2020 / 2021 / 2022 subsets.

In [ ]:
# Load admission-year metadata
patients = pd.read_parquet('../../Data Pre-processing/Preprocessing/data/real_cohort/patients.parquet')
uid_year = patients.set_index('uid')['arrive_yr']

# Map years onto test UIDs — UIDs absent from patients.parquet get NaN
test_years = uid_year.reindex(X_test.index)

n_missing = int(test_years.isna().sum())
print(f'Test UIDs without year info: {n_missing} (included in "All" evaluation only)')

# Create per-year subsets
TEST_YEARS = [2020, 2021, 2022]
X_test_by_year = {yr: X_test[test_years == yr] for yr in TEST_YEARS}
y_test_by_year = {yr: y_test[test_years == yr] for yr in TEST_YEARS}

print('\nTest set year distribution:')
for yr in TEST_YEARS:
    n     = len(X_test_by_year[yr])
    n_pos = int(y_test_by_year[yr]['Outcome'].sum())
    print(f'  {yr}: {n:5d} patients, {n_pos:3d} positive outcomes ({n_pos/n*100:.1f}%)')
n_all = len(X_test)
n_pos_all = int(y_test.Outcome.sum())
print(f'  All : {n_all:5d} patients, {n_pos_all:3d} positive outcomes ({n_pos_all/n_all*100:.1f}%)')

## Loading DAGs

In [ ]:
# Load all adjacency files
adjacency_files = glob.glob("../DAGs/*_adjacency.csv")

dags = {}
dags['Control']     = nx.from_edgelist([(col, 'Outcome') for col in X_train_imp.columns.str.replace(r'(_.+)?$', '', regex=True).unique().tolist() if col != 'Outcome'], create_using=nx.DiGraph())
dags['Correlation'] = nx.from_edgelist([(col, 'Outcome') for col in X_train_corr.columns.str.replace(r'(_.+)?$', '', regex=True).unique().tolist() if col != 'Outcome'], create_using=nx.DiGraph())
for file in adjacency_files:
    dag_name = re.search(r'../DAGs/(.*)_adjacency.csv', file).group(1)
    dag_name = dag_name.replace('x', '$\\cap$')
    dag_name = dag_name.replace(' + ', ' $\\cup$ ')
    dags[dag_name] = nx.DiGraph(pd.read_csv(file, index_col=0))

dags.pop('Golem $\\cap$ PCMB')  # Remove problematic DAG (0 nodes associated with Outcome)
print('DAGs loaded:', list(dags.keys()))

## Helper functions

In [ ]:
from MLstatkit import Bootstrapping

def performance_report(X, y, model):
    y_prob = model.predict_proba(X)[:, 1]

    n_bootstraps = 1000
    ap,        ap_lower,        ap_upper        = Bootstrapping(y.values, y_prob, metric_str='average_precision', n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    auroc,     auroc_lower,     auroc_upper     = Bootstrapping(y.values, y_prob, metric_str='roc_auc',           n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    precision, precision_lower, precision_upper = Bootstrapping(y.values, y_prob, metric_str='precision',         n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    recall,    recall_lower,    recall_upper    = Bootstrapping(y.values, y_prob, metric_str='recall',            n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    f1,        f1_lower,        f1_upper        = Bootstrapping(y.values, y_prob, metric_str='f1',                n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    accuracy,  accuracy_lower,  accuracy_upper  = Bootstrapping(y.values, y_prob, metric_str='accuracy',          n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    ece = ECE(y.values, y_prob.reshape(-1, 1), bins=50)

    return {
        'Average Precision': f"{ap:.2f} ({ap_lower:.2f}, {ap_upper:.2f})",
        'AUROC':             f"{auroc:.2f} ({auroc_lower:.2f}, {auroc_upper:.2f})",
        'Precision':         f"{precision:.2f} ({precision_lower:.2f}, {precision_upper:.2f})",
        'Recall':            f"{recall:.2f} ({recall_lower:.2f}, {recall_upper:.2f})",
        'F1 Score':          f"{f1:.2f} ({f1_lower:.2f}, {f1_upper:.2f})",
        'Accuracy':          f"{accuracy:.2f} ({accuracy_lower:.2f}, {accuracy_upper:.2f})",
        'ECE':               f"{ece:.3f}",
    }

## Training function with year-stratified evaluation
Trains models on the full training set (identical to Experiment 1 in the main notebook) and then evaluates each trained model on:
- The full test set (`Year = 'All'`)
- Each year-split subset (`Year = 2020 / 2021 / 2022`)

In [ ]:
def train_models_year_sensitivity(remove_drugs=False, remove_interventions=False):
    results = []
    preds_by_dag = {}  # {dag_name: {year_label: (y_true, y_prob)}}

    drugs = [
        'Midazolam', 'Fentanyl', 'Olanzapine', 'Haloperidol',
        'Dexmedetomidine', 'Lorazepam', 'Morphine', 'Hydromorphone',
        'Dopamine', 'Cisatracurium', 'Epinephrine', 'Norepinephrine',
        'Milrinone', 'Dobutamine',
    ]
    intervention_nodes = [
        'CRRT Therapy Type', 'ECMO Type', 'Ventilated',
        'Pupillary Reaction', 'Peds Coma Score', 'Endotracheal tube',
    ]

    for dag_name, dag in dags.items():
        if 'Simplified' not in dag_name and dag_name not in ['Control', 'Correlation']:
            continue

        print(f"Processing {dag_name} ({dag.number_of_nodes()} nodes, {dag.number_of_edges()} edges)")

        nodes_in_dag = list(dag.nodes())

        if 'Simplified' in dag_name or dag_name in ['Control', 'Correlation']:
            if remove_drugs:
                nodes_in_dag = [n for n in nodes_in_dag if n not in drugs]
            if remove_interventions:
                nodes_in_dag = [n for n in nodes_in_dag if n not in intervention_nodes]

            if dag_name == 'Correlation':
                features_in_dag = [col for col in X_train_corr.columns
                                   if any(re.search(rf'^{node}(_.+)?$', col) for node in nodes_in_dag)]
            else:
                features_in_dag = [col for col in X_train_imp.columns
                                   if any(re.search(rf'^{node}(_.+)?$', col) for node in nodes_in_dag)]

            n_biomarkers = len(nodes_in_dag) - 1

        else:
            if remove_drugs:
                nodes_in_dag = [n for n in nodes_in_dag if not any(d in n for d in drugs)]
            if remove_interventions:
                nodes_in_dag = [n for n in nodes_in_dag if not any(iv in n for iv in intervention_nodes)]

            features_in_dag = X_train_imp.filter(items=nodes_in_dag).columns.tolist()
            n_biomarkers = (X_train_imp.filter(items=nodes_in_dag)
                            .columns.str.replace(r'(_.+)?$', '', regex=True).nunique() - 1)

        # Train XGBoost
        xgb = xgboost.XGBClassifier(objective='binary:logistic', random_state=42, eval_metric='aucpr')
        xgb.fit(X_train.filter(features_in_dag), y_train.Outcome)

        # Evaluate on full test set and each year subset; store raw predictions for permutation tests
        eval_sets = [('All', X_test.filter(features_in_dag), y_test.Outcome)]
        for yr in TEST_YEARS:
            eval_sets.append((yr, X_test_by_year[yr].filter(features_in_dag), y_test_by_year[yr].Outcome))

        preds_by_dag[dag_name] = {}
        for year_label, X_eval, y_eval in eval_sets:
            y_prob = xgb.predict_proba(X_eval)[:, 1]
            preds_by_dag[dag_name][year_label] = (y_eval.values, y_prob)

            row = {'Model': 'XGB', 'DAG': dag_name, 'Year': year_label,
                   '# Features': len(features_in_dag), '# Biomarkers': n_biomarkers}
            row.update(performance_report(X_eval, y_eval, xgb))
            results.append(row)

    return pd.DataFrame(results), preds_by_dag

## Run year sensitivity experiment
Mirrors Experiment 1 from the main notebook (no drugs or interventions removed) with year-stratified evaluation.

In [ ]:
results_df, preds_by_dag = train_models_year_sensitivity(remove_drugs=False, remove_interventions=False)

In [ ]:
results_df.to_csv('biomarker_counts_fixed/Year Sensitivity.csv', index=False)
results_df

## Summary: AUROC, AUPRC, and Normalized AUPRC by Year
AUPRC depends on class prevalence (π), which differs across years (2020: 14%, 2021: 10%, 2022: 9%).
The **skill score** `(AUPRC − π) / (1 − π)` re-anchors to [0 = random, 1 = perfect] and makes years comparable.
CI bounds transform linearly since π is fixed per year.

In [ ]:
def extract_point_ci(s):
    """Parse 'value (lower, upper)' string into three floats."""
    try:
        s = str(s).replace('(', '').replace(')', '').replace(',', '')
        parts = s.split()
        return float(parts[0]), float(parts[1]), float(parts[2])
    except Exception:
        v = float(s)
        return v, v, v

def extract_point(s):
    return extract_point_ci(s)[0]

def normalize_auprc(auprc_str, pi):
    """Apply skill-score normalization to an AUPRC string with CI."""
    pt, lo, hi = extract_point_ci(auprc_str)
    norm    = (pt - pi) / (1 - pi)
    norm_lo = (lo - pi) / (1 - pi)
    norm_hi = (hi - pi) / (1 - pi)
    return f"{norm:.2f} ({norm_lo:.2f}, {norm_hi:.2f})"

# Per-year prevalence (π) used for normalization
prevalence = {yr: float(y_test_by_year[yr]['Outcome'].mean()) for yr in TEST_YEARS}
prevalence['All'] = float(y_test['Outcome'].mean())
print('Prevalence by year:', {k: f"{v:.3f}" for k, v in prevalence.items()})

# Add normalized AUPRC column
summary = results_df.copy()
summary['Normalized AUPRC'] = summary.apply(
    lambda row: normalize_auprc(row['Average Precision'], prevalence[row['Year']]),
    axis=1
)

# Numeric point estimates for plotting / delta tables
for col in ['Average Precision', 'AUROC', 'Normalized AUPRC']:
    summary[col + '_pt'] = summary[col].apply(extract_point)

# Pivot: rows = DAG, columns = Year
for metric_col, label in [
    ('Average Precision_pt', 'Average Precision (raw)'),
    ('Normalized AUPRC_pt',  'Normalized AUPRC (skill score)'),
    ('AUROC_pt',             'AUROC'),
]:
    pivot = summary.pivot_table(index='DAG', columns='Year', values=metric_col)
    pivot.columns = [str(c) for c in pivot.columns]
    col_order = ['All'] + [str(yr) for yr in TEST_YEARS]
    pivot = pivot[[c for c in col_order if c in pivot.columns]]
    print(f'\n=== {label} ===')
    display(pivot.round(3))

In [ ]:
cols_to_save = ['DAG', 'Year', 'Average Precision', 'Normalized AUPRC', 'AUROC',
                'Precision', 'Recall', 'F1 Score', 'Accuracy', 'ECE',
                '# Features', '# Biomarkers']
summary[cols_to_save].to_csv('biomarker_counts_fixed/Year Sensitivity - Normalized.csv', index=False)

## Plot: AUROC and AUPRC by Year
Line plot across years for each DAG and model type, making any temporal drift immediately visible.

In [ ]:
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'figure.dpi': 150,
    'font.size': 11,
    'lines.linewidth': 1.5,
    'figure.constrained_layout.use': True,
    'pdf.fonttype': 42,
})

year_only = summary[summary['Year'].isin(TEST_YEARS)].copy()
year_only['Year'] = year_only['Year'].astype(int)

metrics_to_plot = [
    ('AUROC_pt',             'AUROC'),
    ('Average Precision_pt', 'Average Precision (raw)'),
    ('Normalized AUPRC_pt',  'Normalized AUPRC (skill score)'),
]

fig, axes = plt.subplots(1, 3)
fig.suptitle('XGB – Performance by Test-Set Year')

for ax, (metric, ylabel) in zip(axes, metrics_to_plot):
    for dag_name, grp in year_only.groupby('DAG'):
        grp_sorted = grp.sort_values('Year')
        ax.plot(grp_sorted['Year'], grp_sorted[metric], marker='o', label=dag_name, alpha=0.7)
    ax.set_xlabel('Year')
    ax.set_ylabel(ylabel)
    ax.set_xticks(TEST_YEARS)
    ax.grid(True, alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.3), fontsize=8, frameon=False)

plt.savefig('biomarker_counts_fixed/Year Sensitivity - XGB.pdf', bbox_inches='tight', dpi=300)
plt.show()
plt.rcParams.update(plt.rcParamsDefault)

## Year-over-year delta table
Shows how much AUROC and Average Precision shift from 2020 → 2021 and 2021 → 2022 for each DAG × Model.

In [ ]:
for metric_col, label in [
    ('AUROC_pt',             'AUROC'),
    ('Average Precision_pt', 'Average Precision (raw)'),
    ('Normalized AUPRC_pt',  'Normalized AUPRC (skill score)'),
]:
    pivot = summary.pivot_table(index='DAG', columns='Year', values=metric_col)
    pivot.columns = [str(c) for c in pivot.columns]
    delta = pd.DataFrame(index=pivot.index)
    delta['2020→2021'] = pivot['2021'] - pivot['2020']
    delta['2021→2022'] = pivot['2022'] - pivot['2021']
    delta['2020→2022'] = pivot['2022'] - pivot['2020']
    print(f'\n=== {label} – year-over-year delta ===')
    display(delta.round(3))

## Permutation test: pairwise year comparisons
Tests whether the same model performs significantly differently across years.

**Design:** pool predictions from years A and B, randomly shuffle year labels (preserving group sizes), recompute the metric difference, repeat 1000 times. The p-value is the fraction of permuted differences with |Δ| ≥ |observed Δ| (two-tailed).

Note: raw AUPRC and AUROC are used inside the permutation loop (not normalized AUPRC), because permuting year labels changes the within-permutation prevalence, making normalization ill-defined. Normalized AUPRC values from the table above provide the prevalence-adjusted magnitudes.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score
from itertools import combinations

def permutation_test_across_years(y_true_A, y_prob_A, y_true_B, y_prob_B,
                                   metric_str='average_precision',
                                   n_permutations=1000, random_state=42):
    """
    Two-tailed permutation test comparing the same model on two year subsets.

    H0: the metric does not differ between year A and year B.
    Permutes year labels (pooling A+B) to build the null distribution of |Δ metric|.
    """
    metric_fn = {'average_precision': average_precision_score,
                 'roc_auc': roc_auc_score}[metric_str]

    obs_A = metric_fn(y_true_A, y_prob_A)
    obs_B = metric_fn(y_true_B, y_prob_B)
    obs_diff = abs(obs_A - obs_B)

    y_true_pool = np.concatenate([y_true_A, y_true_B])
    y_prob_pool = np.concatenate([y_prob_A, y_prob_B])
    n_A = len(y_true_A)

    rng = np.random.default_rng(random_state)
    null_diffs = []
    for _ in range(n_permutations):
        idx = rng.permutation(len(y_true_pool))
        try:
            diff = abs(
                metric_fn(y_true_pool[idx[:n_A]], y_prob_pool[idx[:n_A]])
                - metric_fn(y_true_pool[idx[n_A:]], y_prob_pool[idx[n_A:]])
            )
        except Exception:
            diff = 0.0
        null_diffs.append(diff)

    p_value = float(np.mean(np.array(null_diffs) >= obs_diff))
    return obs_A, obs_B, obs_diff, p_value


# Run pairwise tests for each DAG and metric
year_pairs = list(combinations(TEST_YEARS, 2))
perm_rows = []

for dag_name, year_preds in preds_by_dag.items():
    for yr_A, yr_B in year_pairs:
        y_true_A, y_prob_A = year_preds[yr_A]
        y_true_B, y_prob_B = year_preds[yr_B]

        for metric_str, metric_label in [('average_precision', 'AUPRC'),
                                          ('roc_auc',           'AUROC')]:
            obs_A, obs_B, obs_diff, p_value = permutation_test_across_years(
                y_true_A, y_prob_A, y_true_B, y_prob_B,
                metric_str=metric_str, n_permutations=1000, random_state=42,
            )
            perm_rows.append({
                'DAG':        dag_name,
                'Metric':     metric_label,
                'Year A':     yr_A,
                'Year B':     yr_B,
                f'{metric_label} (A)': round(obs_A, 3),
                f'{metric_label} (B)': round(obs_B, 3),
                '|Δ|':        round(obs_diff, 3),
                'p-value':    round(p_value, 3),
                'Significant (p<0.05)': p_value < 0.05,
            })

perm_df = pd.DataFrame(perm_rows)
perm_df.to_csv('biomarker_counts_fixed/Year Sensitivity - Permutation Tests.csv', index=False)
perm_df

In [ ]:
# Summary: which year pairs show significant differences, and for which DAGs?
print("=== Significant year differences (p < 0.05) ===\n")
for metric_label in ['AUPRC', 'AUROC']:
    sub = perm_df[perm_df['Metric'] == metric_label]
    sig = sub[sub['Significant (p<0.05)']]
    print(f"{metric_label}: {len(sig)} / {len(sub)} comparisons significant")
    if len(sig):
        display(sig[['DAG', 'Year A', 'Year B', '|Δ|', 'p-value']].reset_index(drop=True))
    print()

## Permutation test on AUCNPR (prevalence-normalized)
Repeats the pairwise year comparison on **AUCNPR** instead of raw AUPRC. Unlike the raw test, each year subset is normalized by its **own** prevalence anchor ($\pi$) before the difference is taken, so this is a genuinely distinct test rather than a monotone relabelling — the observed $|\Delta|$ and the permutation null both differ from the raw-AUPRC version.

This addresses the concern that the apparent year-over-year rise in raw AUPRC is largely a prevalence artifact (prevalence falls 14.0% → 10.4% → 9.0% across 2020–2022). Saved to `Year Sensitivity - Permutation AUCNPR.csv`.

In [ ]:
# AUCNPR (prevalence-normalized AUPRC) permutation test.
# Each year subset keeps its OWN prevalence anchor throughout the permutation, so the
# null asks whether prevalence-adjusted discrimination differs between years.
#   AUCNPR   = (AUPRC - AUPR_min) / (1 - AUPR_min)
#   AUPR_min = 1 + (1 - pi) * ln(1 - pi) / pi          (Boyd, Eng & Page, ECML PKDD 2012)

def aucpr_min(pi):
    """Worst-case (minimum achievable) AUPRC at prevalence pi (Boyd et al. 2012)."""
    return 1 + (1 - pi) * np.log(1 - pi) / pi


def permutation_test_aucnpr_across_years(y_true_A, y_prob_A, pi_A,
                                         y_true_B, y_prob_B, pi_B,
                                         n_permutations=1000, random_state=42):
    """
    Two-tailed permutation test on AUCNPR comparing the same model on two year subsets.

    H0: prevalence-adjusted discrimination (AUCNPR) does not differ between year A and B.
    Each side is normalized by its original-year prevalence (pi_A, pi_B); year labels are
    then pooled and shuffled to build the null distribution of |Delta AUCNPR|.
    """
    min_A, min_B = aucpr_min(pi_A), aucpr_min(pi_B)
    npr_A = lambda yt, yp: (average_precision_score(yt, yp) - min_A) / (1 - min_A)
    npr_B = lambda yt, yp: (average_precision_score(yt, yp) - min_B) / (1 - min_B)

    obs_A, obs_B = npr_A(y_true_A, y_prob_A), npr_B(y_true_B, y_prob_B)
    obs_diff = abs(obs_A - obs_B)

    y_true_pool = np.concatenate([y_true_A, y_true_B])
    y_prob_pool = np.concatenate([y_prob_A, y_prob_B])
    n_A = len(y_true_A)

    rng = np.random.default_rng(random_state)
    null_diffs = []
    for _ in range(n_permutations):
        idx = rng.permutation(len(y_true_pool))
        try:
            diff = abs(
                npr_A(y_true_pool[idx[:n_A]], y_prob_pool[idx[:n_A]])
                - npr_B(y_true_pool[idx[n_A:]], y_prob_pool[idx[n_A:]])
            )
        except Exception:
            diff = 0.0
        null_diffs.append(diff)

    p_value = float(np.mean(np.array(null_diffs) >= obs_diff))
    return obs_A, obs_B, obs_diff, p_value


# Run AUCNPR pairwise tests for each DAG (reuses year_pairs from the raw-AUPRC test above)
perm_npr_rows = []
for dag_name, year_preds in preds_by_dag.items():
    for yr_A, yr_B in year_pairs:
        y_true_A, y_prob_A = year_preds[yr_A]
        y_true_B, y_prob_B = year_preds[yr_B]

        obs_A, obs_B, obs_diff, p_value = permutation_test_aucnpr_across_years(
            y_true_A, y_prob_A, prevalence[yr_A],
            y_true_B, y_prob_B, prevalence[yr_B],
            n_permutations=1000, random_state=42,
        )
        perm_npr_rows.append({
            'DAG':        dag_name,
            'Metric':     'AUCNPR',
            'Year A':     yr_A,
            'Year B':     yr_B,
            'AUCNPR (A)': round(obs_A, 3),
            'AUCNPR (B)': round(obs_B, 3),
            '|Δ|':        round(obs_diff, 3),
            'p-value':    round(p_value, 3),
            'Significant (p<0.05)': p_value < 0.05,
        })

perm_npr_df = pd.DataFrame(perm_npr_rows)
perm_npr_df.to_csv('biomarker_counts_fixed/Year Sensitivity - Permutation AUCNPR.csv', index=False)

n_sig = int(perm_npr_df['Significant (p<0.05)'].sum())
print(f"AUCNPR: {n_sig} / {len(perm_npr_df)} comparisons significant (p < 0.05)")
perm_npr_df

# Experiment 2: Vitals & Labs Only (No Drugs or Interventions)
Re-runs the full year-sensitivity pipeline after removing all drug and intervention features, leaving only vitals and laboratory biomarkers (mirrors *Experiment 3: No Drugs or Interventions* in the main notebook). All helper functions, prevalence anchors ($\pi$), and year pairs defined above are reused.

**Note:** once drugs and interventions are removed, some simplified DAGs reduce to identical feature sets (e.g. SCC $\cup$ SPCMB $\equiv$ SPCMB at 131 features; SCC $\equiv$ SCC $\cap$ SPCMB at 93 features), so those rows carry identical metrics.

In [ ]:
# Vitals & labs only: drop drug and intervention features
results_df_vl, preds_by_dag_vl = train_models_year_sensitivity(remove_drugs=True, remove_interventions=True)
results_df_vl.to_csv('biomarker_counts_fixed/Year Sensitivity - Vitals Labs.csv', index=False)
results_df_vl

In [ ]:
# Skill-score and AUCNPR point estimates for the vitals & labs model (reuses helpers above)
summary_vl = results_df_vl.copy()
summary_vl['Normalized AUPRC'] = summary_vl.apply(
    lambda r: normalize_auprc(r['Average Precision'], prevalence[r['Year']]), axis=1)

def aucnpr_point(auprc_str, pi):
    """AUCNPR point estimate from an AUPRC CI string (Boyd et al. 2012)."""
    return (extract_point(auprc_str) - aucpr_min(pi)) / (1 - aucpr_min(pi))

summary_vl['AUCNPR']               = summary_vl.apply(lambda r: aucnpr_point(r['Average Precision'], prevalence[r['Year']]), axis=1)
summary_vl['Average Precision_pt'] = summary_vl['Average Precision'].apply(extract_point)
summary_vl['AUROC_pt']             = summary_vl['AUROC'].apply(extract_point)

cols_to_save = ['DAG', 'Year', 'Average Precision', 'Normalized AUPRC', 'AUROC',
                'Precision', 'Recall', 'F1 Score', 'Accuracy', 'ECE', '# Features', '# Biomarkers']
summary_vl[cols_to_save].to_csv('biomarker_counts_fixed/Year Sensitivity - Vitals Labs - Normalized.csv', index=False)

# Table 1: Raw AUPRC + AUCNPR, wide by year, ordered by AUCNPR on the full test set
raw_piv = summary_vl.pivot_table(index='DAG', columns='Year', values='Average Precision_pt')
npr_piv = summary_vl.pivot_table(index='DAG', columns='Year', values='AUCNPR')
raw_piv.columns = [f'AUPRC_{c}'  for c in raw_piv.columns]
npr_piv.columns = [f'AUCNPR_{c}' for c in npr_piv.columns]
table1_vl = pd.concat([raw_piv, npr_piv], axis=1).sort_values('AUCNPR_All', ascending=False)
table1_vl.to_csv('biomarker_counts_fixed/Year Sensitivity - Vitals Labs - Table1 AUPRC AUCNPR.csv')
table1_vl.round(3)

In [ ]:
# Raw AUPRC / AUROC permutation test — vitals & labs model
perm_rows_vl = []
for dag_name, year_preds in preds_by_dag_vl.items():
    for yr_A, yr_B in year_pairs:
        y_true_A, y_prob_A = year_preds[yr_A]
        y_true_B, y_prob_B = year_preds[yr_B]

        for metric_str, metric_label in [('average_precision', 'AUPRC'),
                                          ('roc_auc',           'AUROC')]:
            obs_A, obs_B, obs_diff, p_value = permutation_test_across_years(
                y_true_A, y_prob_A, y_true_B, y_prob_B,
                metric_str=metric_str, n_permutations=1000, random_state=42,
            )
            perm_rows_vl.append({
                'DAG':        dag_name,
                'Metric':     metric_label,
                'Year A':     yr_A,
                'Year B':     yr_B,
                f'{metric_label} (A)': round(obs_A, 3),
                f'{metric_label} (B)': round(obs_B, 3),
                '|Δ|':        round(obs_diff, 3),
                'p-value':    round(p_value, 3),
                'Significant (p<0.05)': p_value < 0.05,
            })

perm_df_vl = pd.DataFrame(perm_rows_vl)
perm_df_vl.to_csv('biomarker_counts_fixed/Year Sensitivity - Vitals Labs - Permutation Tests.csv', index=False)
for metric_label in ['AUPRC', 'AUROC']:
    sub = perm_df_vl[perm_df_vl['Metric'] == metric_label]
    print(f"{metric_label}: {int(sub['Significant (p<0.05)'].sum())} / {len(sub)} comparisons significant (p < 0.05)")
perm_df_vl

In [ ]:
# AUCNPR (prevalence-normalized) permutation test — vitals & labs model
perm_npr_rows_vl = []
for dag_name, year_preds in preds_by_dag_vl.items():
    for yr_A, yr_B in year_pairs:
        y_true_A, y_prob_A = year_preds[yr_A]
        y_true_B, y_prob_B = year_preds[yr_B]

        obs_A, obs_B, obs_diff, p_value = permutation_test_aucnpr_across_years(
            y_true_A, y_prob_A, prevalence[yr_A],
            y_true_B, y_prob_B, prevalence[yr_B],
            n_permutations=1000, random_state=42,
        )
        perm_npr_rows_vl.append({
            'DAG':        dag_name,
            'Metric':     'AUCNPR',
            'Year A':     yr_A,
            'Year B':     yr_B,
            'AUCNPR (A)': round(obs_A, 3),
            'AUCNPR (B)': round(obs_B, 3),
            '|Δ|':        round(obs_diff, 3),
            'p-value':    round(p_value, 3),
            'Significant (p<0.05)': p_value < 0.05,
        })

perm_npr_df_vl = pd.DataFrame(perm_npr_rows_vl)
perm_npr_df_vl.to_csv('biomarker_counts_fixed/Year Sensitivity - Vitals Labs - Permutation AUCNPR.csv', index=False)

n_sig = int(perm_npr_df_vl['Significant (p<0.05)'].sum())
print(f"AUCNPR: {n_sig} / {len(perm_npr_df_vl)} comparisons significant (p < 0.05)")
perm_npr_df_vl